In [0]:
%sql
CREATE TABLE IF NOT EXISTS pondolens.dev_silver.regional_appropriations (
  gaa_id INT NOT NULL,
  year INT,
  category STRING,
  particular STRING,
  region STRING,
  appropriation_amount DECIMAL(20,2),
  processed_at TIMESTAMP,
  CONSTRAINT regional_appropriations_pk PRIMARY KEY (gaa_id)
)
USING DELTA;

-- Clean text, cast year and remove identical cleaned records.
CREATE OR REPLACE TEMP VIEW regional_gaa_stage AS
SELECT DISTINCT
  CAST(TRIM(CAST(year AS STRING)) AS INT) AS year,
  NULLIF(TRIM(category), '') AS category,
  NULLIF(TRIM(particulars), '') AS particular,

  -- Standardize Region-IVA and Region IV-A to Region IVA.
  CASE
    WHEN TRIM(regions) RLIKE r'(?i)^region[\s-]*([ivx]+)[\s-]*([ab]?)$'
    THEN CONCAT(
      'Region ',
      UPPER(REGEXP_EXTRACT(
        TRIM(regions),
        r'(?i)^region[\s-]*([ivx]+)[\s-]*([ab]?)$',
        1
      )),
      UPPER(REGEXP_EXTRACT(
        TRIM(regions),
        r'(?i)^region[\s-]*([ivx]+)[\s-]*([ab]?)$',
        2
      ))
    )
    WHEN LOWER(TRIM(regions)) = 'nationawide' THEN 'Nationwide'
    ELSE NULLIF(TRIM(regions), '')
  END AS region,

  -- Dashes become zero; blanks stay null. Convert thousand pesos to pesos.
  CASE
    WHEN TRIM(CAST(expenditure_amount AS STRING)) = '-'
      THEN CAST(0 AS DECIMAL(20,2))
    WHEN expenditure_amount IS NULL
      OR TRIM(CAST(expenditure_amount AS STRING)) = ''
      THEN CAST(NULL AS DECIMAL(20,2))
    ELSE CAST(
      CAST(
        REPLACE(TRIM(CAST(expenditure_amount AS STRING)), ',', '')
        AS DECIMAL(20,2)
      ) * 1000
      AS DECIMAL(20,2)
    )
  END AS appropriation_amount
FROM pondolens.dev_bronze.regional_expenditure_gaa;

-- Stop on missing keys or conflicting source records.
SELECT ASSERT_TRUE(
  COUNT(*) = COUNT(year)
  AND COUNT(*) = COUNT(category)
  AND COUNT(*) = COUNT(region)
  AND COUNT(*) = COUNT(
    DISTINCT STRUCT(year, category, particular, region)
  ),
  'Missing key fields or conflicting source records.'
)
FROM regional_gaa_stage;

-- Check that the existing target has unique IDs and composite keys.
SELECT ASSERT_TRUE(
  COUNT(*) = COUNT(DISTINCT gaa_id)
  AND COUNT(*) = COUNT(year)
  AND COUNT(*) = COUNT(category)
  AND COUNT(*) = COUNT(region)
  AND COUNT(*) = COUNT(
    DISTINCT STRUCT(year, category, particular, region)
  )
  AND COUNT_IF(gaa_id <= 1000000) = 0,
  'Missing keys, duplicate records or IDs at or below 1000000.'
)
FROM pondolens.dev_silver.regional_appropriations;

CREATE OR REPLACE TEMP VIEW regional_gaa_numbered AS
WITH new_records AS (
  -- Assign IDs only to records not already in Silver.
  SELECT s.*
  FROM regional_gaa_stage AS s
  LEFT ANTI JOIN pondolens.dev_silver.regional_appropriations AS t
    ON s.year = t.year
    AND s.category = t.category
    AND s.particular <=> t.particular
    AND s.region = t.region
),
last_id AS (
  -- Start at 1000000 for an empty table; otherwise continue from MAX(gaa_id).
  SELECT
    COALESCE(MAX(CAST(gaa_id AS BIGINT)), 1000000) AS last_number
  FROM pondolens.dev_silver.regional_appropriations
)
SELECT
  s.*,
  l.last_number
    + ROW_NUMBER() OVER (
        ORDER BY s.year, s.category, s.particular, s.region
      ) AS next_id
FROM new_records AS s
CROSS JOIN last_id AS l;

-- Ensure new IDs fit the INT column.
SELECT ASSERT_TRUE(
  COUNT(*) = 0,
  'New IDs must be between 1000001 and 2147483647.'
)
FROM regional_gaa_numbered
WHERE next_id NOT BETWEEN 1000001 AND 2147483647;

CREATE OR REPLACE TEMP VIEW regional_gaa_merge_source AS
-- Existing records retain their IDs.
SELECT
  t.gaa_id,
  s.year,
  s.category,
  s.particular,
  s.region,
  s.appropriation_amount
FROM regional_gaa_stage AS s
INNER JOIN pondolens.dev_silver.regional_appropriations AS t
  ON s.year = t.year
  AND s.category = t.category
  AND s.particular <=> t.particular
  AND s.region = t.region

UNION ALL

SELECT
  CAST(next_id AS INT) AS gaa_id,
  year,
  category,
  particular,
  region,
  appropriation_amount
FROM regional_gaa_numbered;

MERGE INTO pondolens.dev_silver.regional_appropriations AS t
USING regional_gaa_merge_source AS s
ON t.year = s.year
  AND t.category = s.category
  AND t.particular <=> s.particular
  AND t.region = s.region

-- Update changed amounts; leave unchanged records untouched.
WHEN MATCHED
  AND NOT (t.appropriation_amount <=> s.appropriation_amount)
THEN UPDATE SET
  t.appropriation_amount = s.appropriation_amount,
  t.processed_at = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
  gaa_id,
  year,
  category,
  particular,
  region,
  appropriation_amount,
  processed_at
)
VALUES (
  s.gaa_id,
  s.year,
  s.category,
  s.particular,
  s.region,
  s.appropriation_amount,
  CURRENT_TIMESTAMP()
);